## Imports

In [1]:
import glob
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

import time

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='processing-2.processing.data.production.internal' ,
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [4]:
local_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/'
local_path

'/Users/E2074/local-datasets/customer/segmentation/consideration/'

In [5]:
city = 'bangalore'
city_name = 'Bangalore'

## Ignore following steps and go to <b>Customer Segment</b> Section directly

## Datasets & Parameter

In [6]:
start_date = datetime.strptime("2025-03-03", "%Y-%m-%d")

run_date = [(start_date + timedelta(weeks=i)).strftime("%Y-%m-%d") for i in range(10)]
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [7]:
def get_segment_customer_base(run_date):
    start_time = time.time()

    ltr_segment = ['HH', 'PHH']
    df_list = []
    
    for segment in ltr_segment:
        base_query = f"""
    
            with base as (
                select 
                    customer_id,
                    taxi_ltr_segment,
                    -- taxi_lifetime_rides,
                    date_diff( 'day', DATE(taxi_lifetime_first_ride_date), DATE(DATE_PARSE(run_date, '%Y-%m-%d'))) rapido_firstride_age
                from 
                    datasets.iallocator_customer_segments
                where 
                    run_date = date_format(date('{run_date}'), '%Y-%m-%d')
                    and taxi_lifetime_last_ride_city = '{city_name}'
                    and taxi_ltr_segment = '{segment}'
                ),
                
                active_week as (
                
                select
                    customerid,
                    91 max_active_days,
                    total_active_days,
                    isAct_w01,
                    isAct_w02,
                    isAct_w03,
                    isAct_w04,
                    isAct_w05,
                    isAct_w06,
                    isAct_w07,
                    isAct_w08,
                    isAct_w09,
                    isAct_w10,
                    isAct_w11,
                    isAct_w12,
                    isAct_w13,
                    13 max_active_weeks,
                    (isAct_w01 + isAct_w02 + isAct_w03 + isAct_w04 + isAct_w05 + isAct_w06 + isAct_w07 + isAct_w08 + isAct_w09 + isAct_w10 + isAct_w11 + isAct_w12 + isAct_w13) total_active_weeks,
                    act_days_w01,
                    act_days_w02,
                    act_days_w03,
                    act_days_w04,
                    act_days_w05,
                    act_days_w06,
                    act_days_w07,
                    act_days_w08,
                    act_days_w09,
                    act_days_w10,
                    act_days_w11,
                    act_days_w12,
                    act_days_w13,
                    net_rides
                from
                    (
                    select
                        customerid,
                        sum(rr_active_day) total_active_days,
                        max(case when day >= date_format(date('{run_date}') - interval '91' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '85' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w01,
                        max(case when day >= date_format(date('{run_date}') - interval '84' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '78' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w02,
                        max(case when day >= date_format(date('{run_date}') - interval '77' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '71' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w03,
                        max(case when day >= date_format(date('{run_date}') - interval '70' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '64' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w04,
                        max(case when day >= date_format(date('{run_date}') - interval '63' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '57' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w05,
                        max(case when day >= date_format(date('{run_date}') - interval '56' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '50' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w06,
                        max(case when day >= date_format(date('{run_date}') - interval '49' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '43' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w07,
                        max(case when day >= date_format(date('{run_date}') - interval '42' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '36' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w08,
                        max(case when day >= date_format(date('{run_date}') - interval '35' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '29' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w09,
                        max(case when day >= date_format(date('{run_date}') - interval '28' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '22' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w10,
                        max(case when day >= date_format(date('{run_date}') - interval '21' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '15' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w11,
                        max(case when day >= date_format(date('{run_date}') - interval '14' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '8' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w12,
                        max(case when day >= date_format(date('{run_date}') - interval '7' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '1' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w13,
                        
                        count(distinct case when day >= date_format(date('{run_date}') - interval '91' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '85' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w01,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '84' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '78' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w02,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '77' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '71' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w03,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '70' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '64' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w04,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '63' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '57' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w05,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '56' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '50' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w06,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '49' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '43' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w07,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '42' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '36' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w08,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '35' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '29' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w09,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '28' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '22' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w10,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '21' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '15' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w11,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '14' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '8' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w12,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '7' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '1' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w13,
                        sum(net_rides_daily) net_rides
                    
                    from    
                        (
                        select 
                            day,
                            customerid, 
                            -- max(fe_sessions_unique_daily) fe_session, 
                            -- sum(rr_sessions_unique_daily) rr_session, 
                            max(case when rr_sessions_unique_daily > 0 then 1 else 0 end) rr_active_day,
                            sum(net_rides_daily) net_rides_daily
                        from 
                            datasets.customer_rf_daily_kpi
                        where 
                            day >= date_format(date('{run_date}') - interval '91' day, '%Y-%m-%d')
                            and day <= date_format(date('{run_date}') - interval '1' day, '%Y-%m-%d')
                            and city = '{city_name}'
                            and service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium', 'Auto NCR')
                            -- and customerid = '6288c6694281e53bbe33d133'
                        group by 1,2
                        )
                    group by 1
                    )
                )
                
                select
                    *
                from
                    base
                left join 
                    active_week
                    on base.customer_id = active_week.customerid
            """

        df_segment = pd.read_sql(base_query, conn4)
        df_segment['run_date'] = run_date
        df_list.append(df_segment)
        print(f"Fetched data for segment: {segment}")
    
    # Concatenate all segment dataframes
    df_base_query = pd.concat(df_list, ignore_index=True)

    df_base_query.to_parquet(local_path + city + '/customer_base_{}.parquet'.format(run_date), index=False)
    end_time = time.time()
    print(f"Executed - Time taken: {end_time - start_time:.4f} seconds")

In [8]:
"""
for i in run_date:
    
    print('Starting - ', i)
    get_segment_customer_base(i)
"""

"\nfor i in run_date:\n    \n    print('Starting - ', i)\n    get_segment_customer_base(i)\n"

In [9]:
# Find all matching parquet files
file_paths = glob.glob(os.path.join(local_path + city + '/customer_base_*.parquet'))

# Dictionary to store dynamically created DataFrames
dfs = {}

for path in file_paths:
    filename = os.path.basename(path)
    # Get last 10 characters of the filename without extension
    name_part = os.path.splitext(filename)[0][-10:]
    # Remove special characters
    cleaned_name = re.sub(r'\W+', '', name_part)
    # Create a dynamic dataframe name
    df_name = f'df_{cleaned_name}'
    # Read the parquet file
    dfs[df_name] = pd.read_parquet(path)

In [10]:
df_list = list(dfs.keys())
df_list

['df_20250414',
 'df_20250331',
 'df_20250317',
 'df_20250428',
 'df_20250407',
 'df_20250421',
 'df_20250505',
 'df_20250310',
 'df_20250324',
 'df_20250303']

## Feature Engineering

In [11]:
def get_feature_engineering(df):
    
    print('feature engineering part')
    # Adding required active month fields
    df['isAct_m01'] = np.where(df[['isAct_w01', 'isAct_w02', 'isAct_w03', 'isAct_w04']].sum(axis=1) >= 1, 1, 0)
    df['isAct_m02'] = np.where(df[['isAct_w05', 'isAct_w06', 'isAct_w07', 'isAct_w08']].sum(axis=1) >= 1, 1, 0)
    df['isAct_m03'] = np.where(df[['isAct_w09', 'isAct_w10', 'isAct_w11', 'isAct_w12', 'isAct_w13']].sum(axis=1) >= 1, 1, 0)
    
    df['max_active_months'] = 3
    df['total_active_months'] = df[['isAct_m01', 'isAct_m02', 'isAct_m03']].sum(axis=1)
    
    df['days_since_1st_ride_in_3months'] = np.where(df['rapido_firstride_age']>=91,91,df['rapido_firstride_age'])
    df['weeks_since_1st_ride_in_3months'] = np.ceil(df['days_since_1st_ride_in_3months']/7)
    df['months_since_1st_ride_in_3months'] = np.ceil(df['days_since_1st_ride_in_3months']/31)

    df["customerid"] = df["customerid"].fillna('NA')
    df.fillna(0,inplace=True)


    # Adding required derived percentage fields for making thresholds
    df['active_days_percet'] = (df['total_active_days']/df['days_since_1st_ride_in_3months']).round(2)
    df['active_weeks_percet'] = (df['total_active_weeks']/df['weeks_since_1st_ride_in_3months']).round(2)
    df['active_months_percet'] = (df['total_active_months']/df['months_since_1st_ride_in_3months']).round(2)

    return df

## Segmentation

In [12]:
def get_segment(df):

    print('segmentation part')
    
    df['hh_monthly_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])) & (df['active_months_percet'] >= 0.66), '1-HH-MONTHLY', None)
    df['hh_quarterly_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])) & (df['total_active_days'] > 0), '2-HH-QUARTERLY', None)
    df['hh_inactive_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])), '3-HH-INACTIVE',  None)
    
    df['daily_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH']))& (df['active_days_percet'] >= 0.45), '4-DAILY', None)
    df['weekly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['active_weeks_percet'] >= 0.70), '5-WEEKLY', None)
    df['monthly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['active_months_percet'] >= 0.66), '6-MONTHLY', None)
    df['quarterly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & df['total_active_days'] > 0, '7-QUARTERLY', None)
    df['inactive_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['total_active_days'] == 0), '8-INACTIVE', None)
    df['new_segment'] = np.where((~df['taxi_ltr_segment'].isin(['HH', 'PHH'])), '9-NEW', None)
    df['segment'] = df['hh_monthly_segment']\
                        .fillna(df['hh_quarterly_segment'])\
                        .fillna(df['hh_inactive_segment'])\
                        .fillna(df['daily_segment'])\
                        .fillna(df['weekly_segment'])\
                        .fillna(df['monthly_segment'])\
                        .fillna(df['quarterly_segment'])\
                        .fillna(df['inactive_segment'])\
                        .fillna(df['new_segment'])\
                        .fillna('check')
    
    df['updated_segment'] = df['segment']
    
    df.loc[((df['updated_segment']=='7-QUARTERLY')& (df['weeks_since_1st_ride_in_3months']<=8)), 'updated_segment'] = '6-MONTHLY'
    df.loc[((df['updated_segment'] == '5-WEEKLY') & (df['weeks_since_1st_ride_in_3months']<=2)), 'updated_segment'] = '6-MONTHLY'

    return df

## Execution

In [13]:
df_list

['df_20250414',
 'df_20250331',
 'df_20250317',
 'df_20250428',
 'df_20250407',
 'df_20250421',
 'df_20250505',
 'df_20250310',
 'df_20250324',
 'df_20250303']

In [14]:
for df_name in df_list:
    print(df_name)
    df = dfs[df_name]
    df_temp = get_feature_engineering(df)
    df_final = get_segment(df_temp)
    df_final.to_parquet(local_path + city + '/segment/{}.parquet'.format(df_name), index=False)
    
    print('---')

df_20250414
feature engineering part
segmentation part
---
df_20250331
feature engineering part
segmentation part
---
df_20250317
feature engineering part
segmentation part
---
df_20250428
feature engineering part
segmentation part
---
df_20250407
feature engineering part
segmentation part
---
df_20250421
feature engineering part
segmentation part
---
df_20250505
feature engineering part
segmentation part
---
df_20250310
feature engineering part
segmentation part
---
df_20250324
feature engineering part
segmentation part
---
df_20250303
feature engineering part
segmentation part
---


## Customer Segment

In [6]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/' + city + '/segment/'

# Get all parquet files in the folder
parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

# Read and concatenate all parquet files into a single DataFrame
df_segment_concat = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)

In [7]:
df_segment_concat.head(1)

,customer_id,taxi_ltr_segment,rapido_firstride_age,customerid,max_active_days,total_active_days,isAct_w01,isAct_w02,isAct_w03,isAct_w04,isAct_w05,isAct_w06,isAct_w07,isAct_w08,isAct_w09,isAct_w10,isAct_w11,isAct_w12,isAct_w13,max_active_weeks,total_active_weeks,act_days_w01,act_days_w02,act_days_w03,act_days_w04,...,act_days_w13,net_rides,run_date,isAct_m01,isAct_m02,isAct_m03,max_active_months,total_active_months,days_since_1st_ride_in_3months,weeks_since_1st_ride_in_3months,months_since_1st_ride_in_3months,active_days_percet,active_weeks_percet,active_months_percet,hh_monthly_segment,hh_quarterly_segment,hh_inactive_segment,daily_segment,weekly_segment,monthly_segment,quarterly_segment,inactive_segment,new_segment,segment,updated_segment
0,655464126305aa25be30b18f,HH,502,NA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2025-03-31,0,0,0,3,0,91,13.0,3.0,0.0,0.0,0.0,None,None,3-HH-INACTIVE,None,None,None,None,None,None,3-HH-INACTIVE,3-HH-INACTIVE


In [8]:
df_segment_concat.shape

(94679458, 58)

In [9]:
df_segment_concat.segment.unique()

array(['3-HH-INACTIVE', '1-HH-MONTHLY', '2-HH-QUARTERLY', '6-MONTHLY',
       '8-INACTIVE', '7-QUARTERLY', '5-WEEKLY', '4-DAILY'], dtype=object)

## Stability

In [10]:
# Reducing df memory 
df_segment_concat = df_segment_concat[['run_date', 'customer_id', 'updated_segment']]

In [11]:
df_segment_concat.run_date.unique()

array(['2025-03-31', '2025-03-17', '2025-04-14', '2025-05-05',
       '2025-04-21', '2025-04-07', '2025-04-28', '2025-03-10',
       '2025-03-03', '2025-03-24'], dtype=object)

In [12]:
df_base = df_segment_concat[df_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base

,run_date,customer_id,updated_segment
0,2025-03-03,65e2ba096e7643720c5b32af,3-HH-INACTIVE
1,2025-03-03,64e6c5eb143ba3ba6f49c168,3-HH-INACTIVE
2,2025-03-03,5d7a9dcda24c96105e4a06ae,3-HH-INACTIVE
3,2025-03-03,657780b9a53a4a14eef9747a,3-HH-INACTIVE
4,2025-03-03,63c652e1926f77641546978e,3-HH-INACTIVE
...,...,...,...
9192848,2025-03-03,62a0921b71bfd2cef5b84bd8,8-INACTIVE
9192849,2025-03-03,653684a5b5145a243b746d25,6-MONTHLY
9192850,2025-03-03,5e25c1416f37f902ba1ad77b,6-MONTHLY
9192851,2025-03-03,64d6d3e25d2459332c79452a,8-INACTIVE


In [13]:
df_stability_window = df_segment_concat[~df_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window

,run_date,customer_id,updated_segment
0,2025-03-10,63e9f56f16de16125fd64cbe,6-MONTHLY
1,2025-03-10,62bd6760ba38cb83dd724c0f,7-QUARTERLY
2,2025-03-10,6239b37e6b6b6492c673a5d6,6-MONTHLY
3,2025-03-10,5fb218a20a41becbbe886ee3,5-WEEKLY
4,2025-03-10,5a89a27af62cc32e4f023917,7-QUARTERLY
...,...,...,...
85486600,2025-05-05,65c320d70753e6514331b97b,8-INACTIVE
85486601,2025-05-05,6770e34c2d02e72cdf707ced,6-MONTHLY
85486602,2025-05-05,60155bdba71b1dbfe3192174,8-INACTIVE
85486603,2025-05-05,61f1fc176916b8680dac3f15,8-INACTIVE


In [14]:
df_stability_window.run_date.unique()

array(['2025-03-10', '2025-03-17', '2025-03-24', '2025-03-31',
       '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28',
       '2025-05-05'], dtype=object)

In [15]:
df_merge = pd.merge(df_base, df_stability_window, how='left', on='customer_id')
df_merge

,run_date_x,customer_id,updated_segment_x,run_date_y,updated_segment_y
0,2025-03-03,65e2ba096e7643720c5b32af,3-HH-INACTIVE,2025-03-10,3-HH-INACTIVE
1,2025-03-03,65e2ba096e7643720c5b32af,3-HH-INACTIVE,2025-03-17,3-HH-INACTIVE
2,2025-03-03,65e2ba096e7643720c5b32af,3-HH-INACTIVE,2025-03-24,3-HH-INACTIVE
3,2025-03-03,65e2ba096e7643720c5b32af,3-HH-INACTIVE,2025-03-31,3-HH-INACTIVE
4,2025-03-03,65e2ba096e7643720c5b32af,3-HH-INACTIVE,2025-04-07,3-HH-INACTIVE
...,...,...,...,...,...
80786646,2025-03-03,63c0dde5291d3ad6ed20eecb,6-MONTHLY,2025-04-07,6-MONTHLY
80786647,2025-03-03,63c0dde5291d3ad6ed20eecb,6-MONTHLY,2025-04-14,6-MONTHLY
80786648,2025-03-03,63c0dde5291d3ad6ed20eecb,6-MONTHLY,2025-04-21,6-MONTHLY
80786649,2025-03-03,63c0dde5291d3ad6ed20eecb,6-MONTHLY,2025-04-28,6-MONTHLY


In [16]:
df_merge_pivot = df_merge.groupby(['run_date_x', 'run_date_y', 'updated_segment_x', 'updated_segment_y']).agg(customers=('customer_id','nunique')).reset_index()

In [17]:
df_merge_pivot

,run_date_x,run_date_y,updated_segment_x,updated_segment_y,customers
0,2025-03-03,2025-03-10,1-HH-MONTHLY,1-HH-MONTHLY,324129
1,2025-03-03,2025-03-10,1-HH-MONTHLY,2-HH-QUARTERLY,73830
2,2025-03-03,2025-03-10,1-HH-MONTHLY,3-HH-INACTIVE,13
3,2025-03-03,2025-03-10,1-HH-MONTHLY,4-DAILY,1642
4,2025-03-03,2025-03-10,1-HH-MONTHLY,5-WEEKLY,3867
...,...,...,...,...,...
413,2025-03-03,2025-05-05,8-INACTIVE,4-DAILY,165
414,2025-03-03,2025-05-05,8-INACTIVE,5-WEEKLY,4
415,2025-03-03,2025-05-05,8-INACTIVE,6-MONTHLY,72037
416,2025-03-03,2025-05-05,8-INACTIVE,7-QUARTERLY,204619


In [18]:
df_merge_pivot.pivot(index = ['run_date_x', 'updated_segment_x', 'updated_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers             \
run_date_y                                     2025-03-10 2025-03-17   
run_date_x updated_segment_x updated_segment_y                         
2025-03-03 1-HH-MONTHLY      1-HH-MONTHLY        324129.0   251211.0   
                             2-HH-QUARTERLY       73830.0   128486.0   
                             3-HH-INACTIVE           13.0       26.0   
                             4-DAILY               1642.0     1227.0   
                             5-WEEKLY              3867.0     8845.0   
                             6-MONTHLY            13473.0    24663.0   
                             7-QUARTERLY            570.0     1523.0   
                             8-INACTIVE               0.0        0.0   
           2-HH-QUARTERLY    1-HH-MONTHLY         33539.0    47177.0   
                             2-HH-QUARTERLY      447579.0   371821.0   
                             3-HH-INACTIVE        54426.0   104452.0   
                             4-DAILY                 17.0       22.0   
                             5-WEEKLY                57.0      209.0   
                             6-MONTHLY             4595.0    10704.0   
                             7-QUARTERLY           5522.0     9238.0   
                             8-INACTIVE              57.0      152.0   
           3-HH-INACTIVE     1-HH-MONTHLY          7232.0     6816.0   
                             2-HH-QUARTERLY       20798.0    38706.0   
                             3-HH-INACTIVE      2738781.0  2709141.0   
                             4-DAILY                515.0      376.0   
                             5-WEEKLY                 0.0        0.0   
                             6-MONTHLY              272.0      842.0   
                             7-QUARTERLY           4817.0    11103.0   
                             8-INACTIVE            1599.0     2499.0   
           4-DAILY           4-DAILY             109889.0   102812.0   
                             5-WEEKLY              7624.0    12750.0   
                             6-MONTHLY             2579.0     4013.0   
                             7-QUARTERLY              1.0        5.0   
                             8-INACTIVE               0.0        0.0   
           5-WEEKLY          4-DAILY               6459.0    10046.0   
                             5-WEEKLY            288129.0   259999.0   
                             6-MONTHLY            33252.0    56019.0   
                             7-QUARTERLY              0.0        1.0   
                             8-INACTIVE               0.0        0.0   
           6-MONTHLY         4-DAILY               1348.0     2654.0   
                             5-WEEKLY             26819.0    44420.0   
                             6-MONTHLY          1765441.0  1637277.0   
                             7-QUARTERLY         150685.0   242892.0   
                             8-INACTIVE               2.0        2.0   
           7-QUARTERLY       4-DAILY                  1.0       42.0   
                             5-WEEKLY                 0.0        0.0   
                             6-MONTHLY           122277.0   197211.0   
                             7-QUARTERLY         756367.0   601166.0   
                             8-INACTIVE           75572.0   142663.0   
           8-INACTIVE        4-DAILY                  6.0        3.0   
                             5-WEEKLY                 0.0        0.0   
                             6-MONTHLY              129.0      136.0   
                             7-QUARTERLY          41686.0    74881.0   
                             8-INACTIVE         1990760.0  1949874.0   

                                                                      \
run_date_y                                     2025-03-24 2025-03-31   
run_date_x updated_segment_x updated_segment_y                         
2025-03-03 1-HH-MONTHLY      1-HH-MONTHLY        197074.0   159057.0   
               

In [19]:
df_base.groupby(['run_date', 'updated_segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,updated_segment,customers
0,2025-03-03,1-HH-MONTHLY,419656
1,2025-03-03,2-HH-QUARTERLY,548198
2,2025-03-03,3-HH-INACTIVE,2778987
3,2025-03-03,4-DAILY,122112
4,2025-03-03,5-WEEKLY,333348
5,2025-03-03,6-MONTHLY,1973724
6,2025-03-03,7-QUARTERLY,974298
7,2025-03-03,8-INACTIVE,2042530


## Previous Segment Stability

### taxi_regularity_segment

In [58]:
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [75]:
def get_previous_segment_customer_base(run_date):
    
    base_query = f"""
    
    select 
        run_date, 
        customer_id, 
        taxi_regularity_segment
        -- taxi_fe_regularity_segment 
    from 
        datasets.iallocator_customer_segments 
    where 
        run_date = date_format(date('{run_date}'), '%Y-%m-%d')
        and taxi_lifetime_last_ride_city = 'Bangalore'
        and taxi_regularity_segment != 'UNKNOWN' 
    """
    
    df_base_query = pd.read_sql(base_query, conn2)
    df_base_query['run_date'] = run_date
    print('executed - ', run_date)

    df_base_query.to_parquet(local_path + '/prev_segment/prev_customer_base_{}.parquet'.format(run_date), index=False)

In [76]:
for i in run_date[2:]:
    
    print('starting - ', i)
    get_previous_segment_customer_base(i)

starting -  2025-03-17
executed -  2025-03-17
starting -  2025-03-24
executed -  2025-03-24
starting -  2025-03-31
executed -  2025-03-31
starting -  2025-04-07
executed -  2025-04-07
starting -  2025-04-14
executed -  2025-04-14
starting -  2025-04-21
executed -  2025-04-21
starting -  2025-04-28
executed -  2025-04-28
starting -  2025-05-05
executed -  2025-05-05


In [79]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/stability/prev_segment'

# Get all parquet files in the folder
prev_parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

# Read and concatenate all parquet files into a single DataFrame
df_previous_segment_concat = pd.concat([pd.read_parquet(file) for file in prev_parquet_files], ignore_index=True)

In [80]:
df_previous_segment_concat.head(1)

,run_date,customer_id,taxi_regularity_segment
0,2025-03-31,5b8a7049d8073f681de07441,RARE_NEED


In [81]:
df_previous_segment_concat.shape

(56364524, 3)

### Stability

In [84]:
df_previous_segment_concat.run_date.unique()

array(['2025-03-31', '2025-03-17', '2025-04-14', '2025-04-28',
       '2025-04-07', '2025-04-21', '2025-03-10', '2025-05-05',
       '2025-03-03', '2025-03-24'], dtype=object)

In [82]:
df_base_prev = df_previous_segment_concat[df_previous_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base_prev

,run_date,customer_id,taxi_regularity_segment
0,2025-03-03,6454d93a62374ed412653df1,DAILY
1,2025-03-03,640ab57a9dba1afb507ad37d,MONTHLY
2,2025-03-03,5d46f66a8dbbed6b854b9411,RARE_NEED
3,2025-03-03,672e01c57a90cba7fd4bb026,BI_WEEKLY
4,2025-03-03,657bf4a4f275491ea5f8a2cd,RARE_NEED
...,...,...,...
5446007,2025-03-03,677bc089f30833be453a1ee8,DAILY
5446008,2025-03-03,6717be255763f6f606549a8c,WEEKLY
5446009,2025-03-03,62b4463d68ed98eb74b1a8c8,RARE_NEED
5446010,2025-03-03,6688b91bf4352e8b9471f06d,MONTHLY


In [83]:
df_stability_window_prev = df_previous_segment_concat[~df_previous_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window_prev

,run_date,customer_id,taxi_regularity_segment
0,2025-03-10,623c8014b9a21e17a1ccffb3,WEEKLY
1,2025-03-10,634fdfc14282cecc855b269e,RARE_NEED
2,2025-03-10,5c917e3a8c352421eaf2d740,RARE_NEED
3,2025-03-10,5c4e8cc64a267149c77d6266,RARE_NEED
4,2025-03-10,6221c6afc1a4e4690d63894b,BI_WEEKLY
...,...,...,...
50918507,2025-05-05,62316b5c3a5bedb28fbeaa16,RARE_NEED
50918508,2025-05-05,5c1c2fe2e9ba5b22438983a5,RARE_NEED
50918509,2025-05-05,634bd990fe45b5d3814353e4,RARE_NEED
50918510,2025-05-05,5c931cc08c352421eaf4a4c2,WEEKLY


In [85]:
df_stability_window_prev.run_date.unique()

array(['2025-03-10', '2025-03-17', '2025-03-24', '2025-03-31',
       '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28',
       '2025-05-05'], dtype=object)

In [87]:
df_merge_prev = pd.merge(df_base_prev, df_stability_window_prev, how='left', on='customer_id')
df_merge_prev

,run_date_x,customer_id,taxi_regularity_segment_x,run_date_y,taxi_regularity_segment_y
0,2025-03-03,6454d93a62374ed412653df1,DAILY,2025-03-10,DAILY
1,2025-03-03,6454d93a62374ed412653df1,DAILY,2025-03-17,DAILY
2,2025-03-03,6454d93a62374ed412653df1,DAILY,2025-03-24,DAILY
3,2025-03-03,6454d93a62374ed412653df1,DAILY,2025-03-31,DAILY
4,2025-03-03,6454d93a62374ed412653df1,DAILY,2025-04-07,DAILY
...,...,...,...,...,...
47414424,2025-03-03,64631423718a270a602b5795,MONTHLY,2025-04-07,BI_WEEKLY
47414425,2025-03-03,64631423718a270a602b5795,MONTHLY,2025-04-14,BI_WEEKLY
47414426,2025-03-03,64631423718a270a602b5795,MONTHLY,2025-04-21,BI_WEEKLY
47414427,2025-03-03,64631423718a270a602b5795,MONTHLY,2025-04-28,BI_WEEKLY


In [88]:
df_merge_pivot_prev = df_merge_prev.groupby(['run_date_x', 'run_date_y', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y']).agg(customers=('customer_id','nunique')).reset_index()
df_merge_pivot_prev

,run_date_x,run_date_y,taxi_regularity_segment_x,taxi_regularity_segment_y,customers
0,2025-03-03,2025-03-10,BI_WEEKLY,BI_WEEKLY,699039
1,2025-03-03,2025-03-10,BI_WEEKLY,DAILY,5764
2,2025-03-03,2025-03-10,BI_WEEKLY,MONTHLY,112066
3,2025-03-03,2025-03-10,BI_WEEKLY,WEEKLY,68657
4,2025-03-03,2025-03-10,DAILY,BI_WEEKLY,4167
5,2025-03-03,2025-03-10,DAILY,DAILY,229567
6,2025-03-03,2025-03-10,DAILY,WEEKLY,13761
7,2025-03-03,2025-03-10,MONTHLY,BI_WEEKLY,96079
8,2025-03-03,2025-03-10,MONTHLY,MONTHLY,602270
9,2025-03-03,2025-03-10,MONTHLY,RARE_NEED,101734


In [89]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers  \
run_date_y                                                     2025-03-10   
run_date_x taxi_regularity_segment_x taxi_regularity_segment_y              
2025-03-03 BI_WEEKLY                 BI_WEEKLY                   699039.0   
                                     DAILY                         5764.0   
                                     MONTHLY                     112066.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       68657.0   
           DAILY                     BI_WEEKLY                     4167.0   
                                     DAILY                       229567.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       13761.0   
           MONTHLY                   BI_WEEKLY                    96079.0   
                                     DAILY                            0.0   
                                     MONTHLY                     602270.0   
                                     RARE_NEED                   101734.0   
                                     WEEKLY                           0.0   
           RARE_NEED                 BI_WEEKLY                        0.0   
                                     DAILY                            0.0   
                                     MONTHLY                      74919.0   
                                     RARE_NEED                  2687514.0   
                                     WEEKLY                           0.0   
           WEEKLY                    BI_WEEKLY                    68656.0   
                                     DAILY                        14488.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                      600345.0   

                                                                           \
run_date_y                                                     2025-03-17   
run_date_x taxi_regularity_segment_x taxi_regularity_segment_y              
2025-03-03 BI_WEEKLY                 BI_WEEKLY                   585691.0   
                                     DAILY                        11095.0   
                                     MONTHLY                     178838.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       99802.0   
           DAILY                     BI_WEEKLY                     7460.0   
                                     DAILY                       216499.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       21478.0   
           MONTHLY                   BI_WEEKLY                   132910.0   
                                     DAILY                            0.0   
                                     MONTHLY                     481937.0   
                                     RARE_NEED                   177401.0   
                                     WEEKLY                           0.0   
           RARE_NEED                 BI_WEEKLY                    14103.0   
                                     DAILY                            0.0   
                                     MONTHLY                     115143.0   
                                     RARE_NEED                  2620906.0   
                                     WEEKLY                           0.0   
           WEEKLY                    BI_WEEKLY                    97050.0   
                                     DAILY                 

In [91]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0).to_clipboard(index=False)

In [92]:
df_base_prev.groupby(['run_date', 'taxi_regularity_segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,taxi_regularity_segment,customers
0,2025-03-03,BI_WEEKLY,901613
1,2025-03-03,DAILY,254588
2,2025-03-03,MONTHLY,810794
3,2025-03-03,RARE_NEED,2777777
4,2025-03-03,WEEKLY,701240


### FE Regularity

In [6]:
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [7]:
local_path

'/Users/E2074/local-datasets/customer/segmentation/consideration/stability2'

In [26]:
def get_previous_segment_customer_base(run_date):
    
    base_query = f"""
    
    select 
        run_date, 
        customer_id, 
        -- taxi_regularity_segment
        taxi_fe_regularity_segment 
    from 
        datasets.iallocator_customer_segments 
    where 
        run_date = date_format(date('{run_date}'), '%Y-%m-%d')
        and taxi_lifetime_last_ride_city = 'Bangalore'
        and taxi_fe_regularity_segment != 'UNKNOWN' 
    """
    
    df_base_query = pd.read_sql(base_query, conn4)
    df_base_query['run_date'] = run_date
    print('executed - ', run_date)

    df_base_query.to_parquet(local_path + '/prev_segment/prev_customer_base_{}.parquet'.format(run_date), index=False)

In [27]:
for i in run_date[3:]:
    
    print('starting - ', i)
    get_previous_segment_customer_base(i)

starting -  2025-03-24
executed -  2025-03-24
starting -  2025-03-31
executed -  2025-03-31
starting -  2025-04-07
executed -  2025-04-07
starting -  2025-04-14
executed -  2025-04-14
starting -  2025-04-21
executed -  2025-04-21
starting -  2025-04-28
executed -  2025-04-28
starting -  2025-05-05
executed -  2025-05-05


In [28]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/stability2/prev_segment'

# Get all parquet files in the folder
prev_parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

# Read and concatenate all parquet files into a single DataFrame
df_previous_segment_concat = pd.concat([pd.read_parquet(file) for file in prev_parquet_files], ignore_index=True)

In [29]:
df_previous_segment_concat.head(1)

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-31,6474a5c8694393cc7e442e77,DAILY


In [30]:
df_previous_segment_concat.shape

(35532535, 3)

### Stability

In [31]:
df_previous_segment_concat.run_date.unique()

array(['2025-03-31', '2025-03-17', '2025-04-14', '2025-04-28',
       '2025-04-07', '2025-04-21', '2025-03-10', '2025-05-05',
       '2025-03-03', '2025-03-24'], dtype=object)

In [32]:
df_base_prev = df_previous_segment_concat[df_previous_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base_prev

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY
1,2025-03-03,61dbe366fb8e4d610931ab58,BI_WEEKLY
2,2025-03-03,661a4d851f79f98055656dc3,MONTHLY
3,2025-03-03,5e33d682c725ca6c303e47e2,RARE_NEED
4,2025-03-03,61619a34aa0ecd1f9713558f,BI_WEEKLY
...,...,...,...
3448101,2025-03-03,65d0a2f26e764329106ed317,MONTHLY
3448102,2025-03-03,66384d8d5b3860598dc602e1,BI_WEEKLY
3448103,2025-03-03,5d844ff1af794421566a1ef1,WEEKLY
3448104,2025-03-03,6585c70a1093bb34fc3a09a0,WEEKLY


In [33]:
df_stability_window_prev = df_previous_segment_concat[~df_previous_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window_prev

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-10,5abdecea25abd634611d51aa,MONTHLY
1,2025-03-10,62a735f41420e29d95401dc1,MONTHLY
2,2025-03-10,67aa9d57fe551003ad9fe5d9,MONTHLY
3,2025-03-10,67c341027c3ccf7a49876869,RARE_NEED
4,2025-03-10,6730dedc8ea7d581ee78aa92,BI_WEEKLY
...,...,...,...
32084424,2025-05-05,67e38cabd25c76821790bfd1,RARE_NEED
32084425,2025-05-05,6583d607cfe5392da9d31334,WEEKLY
32084426,2025-05-05,639b49dbd79523ee6cce4771,WEEKLY
32084427,2025-05-05,647dd4970b7c4a5f730081c2,RARE_NEED


In [34]:
df_stability_window_prev.run_date.unique()

array(['2025-03-10', '2025-03-17', '2025-03-24', '2025-03-31',
       '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28',
       '2025-05-05'], dtype=object)

In [35]:
df_merge_prev = pd.merge(df_base_prev, df_stability_window_prev, how='left', on='customer_id')
df_merge_prev

,run_date_x,customer_id,taxi_fe_regularity_segment_x,run_date_y,taxi_fe_regularity_segment_y
0,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-03-10,BI_WEEKLY
1,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-03-17,BI_WEEKLY
2,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-03-24,BI_WEEKLY
3,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-03-31,MONTHLY
4,2025-03-03,64e70e90be1489e998c46848,BI_WEEKLY,2025-04-07,MONTHLY
...,...,...,...,...,...
26076936,2025-03-03,61e5682e93bdd008c8df9947,BI_WEEKLY,2025-04-07,BI_WEEKLY
26076937,2025-03-03,61e5682e93bdd008c8df9947,BI_WEEKLY,2025-04-14,MONTHLY
26076938,2025-03-03,61e5682e93bdd008c8df9947,BI_WEEKLY,2025-04-21,MONTHLY
26076939,2025-03-03,61e5682e93bdd008c8df9947,BI_WEEKLY,2025-04-28,MONTHLY


In [36]:
df_merge_pivot_prev = df_merge_prev.groupby(['run_date_x', 'run_date_y', 'taxi_fe_regularity_segment_x', 'taxi_fe_regularity_segment_y']).agg(customers=('customer_id','nunique')).reset_index()
df_merge_pivot_prev

,run_date_x,run_date_y,taxi_fe_regularity_segment_x,taxi_fe_regularity_segment_y,customers
0,2025-03-03,2025-03-10,BI_WEEKLY,BI_WEEKLY,852070
1,2025-03-03,2025-03-10,BI_WEEKLY,DAILY,9595
2,2025-03-03,2025-03-10,BI_WEEKLY,MONTHLY,119637
3,2025-03-03,2025-03-10,BI_WEEKLY,WEEKLY,55537
4,2025-03-03,2025-03-10,DAILY,BI_WEEKLY,6707
5,2025-03-03,2025-03-10,DAILY,DAILY,263169
6,2025-03-03,2025-03-10,DAILY,WEEKLY,14437
7,2025-03-03,2025-03-10,MONTHLY,BI_WEEKLY,119766
8,2025-03-03,2025-03-10,MONTHLY,MONTHLY,782052
9,2025-03-03,2025-03-10,MONTHLY,RARE_NEED,101823


In [37]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_fe_regularity_segment_x', 'taxi_fe_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers  \
run_date_y                                                           2025-03-10   
run_date_x taxi_fe_regularity_segment_x taxi_fe_regularity_segment_y              
2025-03-03 BI_WEEKLY                    BI_WEEKLY                      852070.0   
                                        DAILY                            9595.0   
                                        MONTHLY                        119637.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          55537.0   
           DAILY                        BI_WEEKLY                        6707.0   
                                        DAILY                          263169.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          14437.0   
           MONTHLY                      BI_WEEKLY                      119766.0   
                                        DAILY                               0.0   
                                        MONTHLY                        782052.0   
                                        RARE_NEED                      101823.0   
                                        WEEKLY                              0.0   
           RARE_NEED                    BI_WEEKLY                           0.0   
                                        DAILY                               0.0   
                                        MONTHLY                         68392.0   
                                        RARE_NEED                      425258.0   
                                        WEEKLY                              0.0   
           WEEKLY                       BI_WEEKLY                       55659.0   
                                        DAILY                           13805.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                         380275.0   

                                                                                 \
run_date_y                                                           2025-03-17   
run_date_x taxi_fe_regularity_segment_x taxi_fe_regularity_segment_y              
2025-03-03 BI_WEEKLY                    BI_WEEKLY                      744171.0   
                                        DAILY                           17325.0   
                                        MONTHLY                        179325.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          84522.0   
           DAILY                        BI_WEEKLY                       11878.0   
                                        DAILY                          247903.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          22008.0   
           MONTHLY                      BI_WEEKLY                      180067.0   
                                        DAILY                               0.0   
                                        MONTHLY                        621075.0   
                                        RARE_NEED                      153928.0   
                                        WEEKLY                              0.0   
           RARE_NEED                    BI_WEEKLY                         486.0   
                                        DAILY                               0.0   
                                        MONTHLY                         98306.0   
  

In [38]:
df_base_prev.groupby(['run_date', 'taxi_fe_regularity_segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,taxi_fe_regularity_segment,customers
0,2025-03-03,BI_WEEKLY,1056482
1,2025-03-03,DAILY,292637
2,2025-03-03,MONTHLY,1047096
3,2025-03-03,RARE_NEED,590726
4,2025-03-03,WEEKLY,461165
